In [1]:
import json
from datetime import datetime, timedelta
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from anthropic.types import Message, ToolParam

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })

def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    return client.messages.create(**params)

def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

In [3]:
save_article_schema = {
    "name": "save_article",
    "description": "Saves an article with its abstract and metadata. Use it when the user asks to save or store an article. Returns a confirmation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {"type": "string", "description": "A short abstract of the article."},
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {"type": "integer", "description": "Number of words."},
                    "review": {"type": "string", "description": "A one-sentence review."},
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
}

def save_article(abstract, meta):
    return f"Saved article ({meta['word_count']} words)."

messages = [{
    "role": "user",
    "content": "Write a short article about electric cars and save it with save_article.",
}]

## A helper that streams and times the chunks

In [4]:
import time

def stream_tool_args(tools, use_beta=False):
    start = time.time()
    snapshot = None
    streamer = client.beta.messages.stream if use_beta else client.messages.stream

    with streamer(model=model, max_tokens=1500, messages=messages, tools=tools) as stream:
        for event in stream:
            if event.type == "input_json":
                print(f"[{time.time() - start:5.1f}s] {event.partial_json!r}")
                snapshot = event.snapshot
        final = stream.get_final_message()
    return snapshot, final

Each line shows when the chunk arrived, so you can see the buffering difference.

## Normal streaming

In [5]:
snapshot, final = stream_tool_args([save_article_schema])
print("\nFinal snapshot:", snapshot)

[  1.0s] ''
[  1.9s] '{"abstr'
[  1.9s] 'act": "'
[  1.9s] 'Electri'
[  1.9s] 'c vehic'
[  1.9s] 'les (EV'
[  1.9s] 's) are '
[  1.9s] 'transfo'
[  1.9s] 'rming t'
[  1.9s] 'he auto'
[  1.9s] 'motive '
[  1.9s] 'industr'
[  1.9s] 'y with '
[  1.9s] 'zero ta'
[  1.9s] 'ilpipe '
[  1.9s] 'emissio'
[  1.9s] 'ns and '
[  1.9s] 'lower o'
[  1.9s] 'peratin'
[  1.9s] 'g costs'
[  1.9s] '. As ba'
[  1.9s] 'ttery t'
[  1.9s] 'echnolo'
[  1.9s] 'gy impr'
[  1.9s] 'oves an'
[  1.9s] 'd charg'
[  1.9s] 'ing inf'
[  1.9s] 'rastruc'
[  1.9s] 'ture ex'
[  1.9s] 'pands g'
[  1.9s] 'lobally'
[  1.9s] ', elect'
[  1.9s] 'ric car'
[  1.9s] 's are b'
[  1.9s] 'ecoming'
[  1.9s] ' increa'
[  1.9s] 'singly '
[  1.9s] 'practic'
[  1.9s] 'al and '
[  1.9s] 'afforda'
[  1.9s] 'ble for'
[  1.9s] ' everyd'
[  1.9s] 'ay driv'
[  1.9s] 'ers. Ma'
[  1.9s] 'jor aut'
[  1.9s] 'omakers'
[  2.0s] ' are in'
[  2.0s] 'vesting'
[  2.0s] ' billio'
[  2.0s] 'ns in E'
[  2.0s] 'V devel'
[  2.0s] 'opment,'
[  2.0s] ' signal'


## Fine-grained streaming

In [6]:
fine_tool = {**save_article_schema, "eager_input_streaming": True}

snapshot, final = stream_tool_args([fine_tool], use_beta=True)
print("\nFinal snapshot:", snapshot)

[  1.0s] ''
[  1.1s] '{"abstract": "Electric cars'
[  1.1s] ' are revolut'
[  1.2s] 'ionizing transportation'
[  1.2s] ' by'
[  1.2s] ' offering a'
[  1.2s] ' clean'
[  1.2s] ', efficient'
[  1.3s] ' alternative to traditional gas'
[  1.3s] 'oline vehicles.'
[  1.3s] ' Powered by rech'
[  1.3s] 'argeable batteries,'
[  1.4s] ' EV'
[  1.4s] 's produce zero'
[  1.4s] ' direct'
[  1.4s] ' emissions and significantly'
[  1.5s] ' reduce carbon'
[  1.5s] ' footprints.'
[  1.5s] ' With'
[  1.5s] ' improving'
[  1.6s] ' battery technology,'
[  1.6s] ' expanding'
[  1.6s] ' charging infrastructure, and'
[  1.6s] ' decre'
[  1.7s] 'asing costs, electric'
[  1.7s] ' vehicles'
[  1.7s] ' are becoming increasingly accessible'
[  1.7s] ' to'
[  1.8s] ' consumers worldwide. Major'
[  1.8s] ' automakers are investing'
[  1.8s] ' heavily in E'
[  1.8s] 'V development'
[  1.9s] ', making'
[  1.9s] ' them the'
[  1.9s] ' future of sustainable'
[  1.9s] ' mobility'
[  2.0s] '.'
[  2.1s] '", "meta": {\n  "

## Parse safely, then run the tool

In [7]:
def parse_args(snapshot):
    if isinstance(snapshot, dict):
        return snapshot
    try:
        return json.loads(snapshot)
    except (json.JSONDecodeError, TypeError):
        return None

args = parse_args(snapshot)

if args is None:
    print("Invalid JSON received. Retry or repair before running the tool.")
else:
    print(save_article(**args))

Saved article (75 words).


If the arguments are invalid, the tool is not run. That is the core safety rule of this lesson.